In [1]:
import os
import re
from dotenv import load_dotenv
import json
from typing import TypedDict, Callable
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, AgentMiddleware
from langchain.messages import SystemMessage, HumanMessage
from prompt import system_prompt

import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

load_dotenv()

def extract_id(file_name):
    match = re.search(r'\d+', file_name)
    return int(match.group()) if match else 0

class Facet(TypedDict):
    name: str
    description: str
    content: str

def simulate_persona(pid: int, input: str, facets: list[Facet]):
    @tool
    def retrieve_persona_facet(facet_name: str) -> str:
        """Retrieve the content of a specific facet of the persona, which includes historical responses to a series of questions. 

        Args:
            facet_name: The name of the facet to retrieve.
        """
        for facet in facets:
            if facet["name"] == facet_name:
                return f"Retrieved Facet: {facet_name}\n\nContent: {facet['content']}"

        available = ", ".join(f["name"] for f in facets)
        return f"Facet '{facet_name}' not found. Available facets for this persona: {available}"

    class FacetMiddleware(AgentMiddleware):        
        tools = [retrieve_persona_facet]
        
        def __init__(self):
            facet_list = []
            for facet in facets:
                facet_list.append(
                    f"- **{facet['name']}**: {facet['description']}"
                )
            self.facets_prompt = "\n".join(facet_list)

        def wrap_model_call(
            self,
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse],
        ) -> ModelResponse:
            facets_addendum = (
                f"\n\n## Available Facets\n\n{self.facets_prompt}\n\n"
                "Use the 'retrieve_persona_facet' tool when you need detailed information "
                "about the persona."
            )

            new_content = list(request.system_message.content_blocks) + [
                {"type": "text", "text": facets_addendum}
            ]
            new_system_message = SystemMessage(content=new_content)
            modified_request = request.override(system_message=new_system_message)
            return handler(modified_request)
    
    model = init_chat_model(model="deepseek-chat")

    conn = sqlite3.connect("conversations.db", check_same_thread=False)
    checkpointer = SqliteSaver(conn)

    agent = create_agent(
        model,
        system_prompt=SystemMessage(system_prompt),
        middleware=[FacetMiddleware()],
        checkpointer=checkpointer,
    )

    thread_id = str(pid)
    config = {"configurable": {"thread_id": thread_id}}

    result = agent.invoke(
        {"messages": [HumanMessage(input)]},
        config
    )

    return result

In [2]:
input_files = sorted(os.listdir("text_questions"), key=extract_id)
skills_files = sorted(os.listdir("skills"), key=extract_id)

input_list = []
skills_list = []

for input_file, skills_file in zip(input_files, skills_files):
    with open(f"text_questions/{input_file}", "r", encoding="utf-8") as f:
        input_list.append(f.read())
    with open(f"skills/{skills_file}", "r", encoding="utf-8") as f:
        skills_list.append(json.load(f))

In [3]:
pid = 1
for input, skills in zip(input_list, skills_list):
    response = simulate_persona(pid, input, skills)
    pid += 1
    break

In [4]:
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

Please answer the following questions as if you were taking this survey. The expected output is a JSON object and the format is provided in the end. 
---

Q1:
Would you support or oppose...
Question Type: Matrix
Options:
  1 = Strongly oppose
  2 = Somewhat oppose
  3 = Neither oppose nor support
  4 = Somewhat support
  5 = Strongly support
1. Placing a tax on carbon emissions?
Answer: [Masked]
2. Ensuring 40% of all new clean energy infrastructure development spending goes to low-income communities?
Answer: [Masked]
3. Federal investments to ensure a carbon-pollution free electricity sector by 2035?
Answer: [Masked]
4. A ‘Medicare for All’ system in which all Americans would get healthcare from a government-run plan?
Answer: [Masked]
5. A ‘public option’, which would allow Americans to buy into a government-run healthcare plan if they choose to do so?
Answer: [Masked]
6. Immigration reforms that would p